# Genome Repair Console -- Local WSL Evaluation & Benchmarking

This notebook has been adapted for local execution under Windows Subsystem for Linux (WSL). Kaggle-specific file system workarounds, apt-get commands, and custom tool compilation steps have been removed. All required external bioinformatics tools (`minimap2`, `flye`, `racon`, `medaka`, `pomoxis`, `mummer4`, `rasusa`) are managed via your active Conda environment.

**Structure:**
- **Part A**: `evaluate.py` on a fresh held-out set + `variant_preservation_test.py`. Self-contained evaluation using the local checkpoint.
- **Part B**: Full Flye -> Racon -> Medaka baseline comparison via `run_comparison.sh`.

## 1. Environment & Dependency Verification

In [1]:
import importlib
import subprocess
import sys

def check_python_pkg(pkg):
    try:
        importlib.import_module(pkg)
        print(f"[OK] Python package '{pkg}' is available.")
    except ImportError:
        print(f"[MISSING] Python package '{pkg}' not found in active environment!")

for pkg in ['edlib', 'mappy', 'fastapi', 'badread', 'torch', 'pytest']:
    check_python_pkg(pkg)

def check_cli_tool(cmd):
    res = subprocess.run(['which', cmd], capture_output=True, text=True)
    if res.returncode == 0:
        print(f"[OK] Binary '{cmd}' found at: {res.stdout.strip()}")
    else:
        print(f"[MISSING] Binary '{cmd}' not found in PATH!")

check_cli_tool('minimap2')

print(f"[INFO] Python executable: {sys.executable}")
print("[INFO] Should contain 'genome_repair' if this kernel matches your activated conda env.")

if torch.cuda.is_available():
    print(f"[OK] CUDA available -- device: {torch.cuda.get_device_name(0)}")
else:
    print("[WARNING] CUDA not available -- this run will use CPU. If you expect a GPU, "
          "check `nvidia-smi` from inside WSL and confirm this env's torch build is CUDA, "
          "not CPU-only (`python -c 'import torch; print(torch.__version__)'`).")

[OK] Python package 'edlib' is available.
[OK] Python package 'mappy' is available.
[OK] Python package 'fastapi' is available.
[OK] Python package 'badread' is available.
[OK] Python package 'torch' is available.
[OK] Python package 'pytest' is available.
[OK] Binary 'minimap2' found at: /home/syed_muksid/miniconda3/envs/genome_repair/bin/minimap2
[INFO] Python executable: /home/syed_muksid/miniconda3/envs/genome_repair/bin/python
[INFO] Should contain 'genome_repair' if this kernel matches your activated conda env.


NameError: name 'torch' is not defined

## 2. Confirm Checkpoint and Reference Genomes

Verifying that `checkpoints/model_best.pt`, `data/reference/ecoli_k12_mg1655.fasta`, and `data/reference/ecoli_zymo_benchmark_strain.fasta` exist relative to the project root.

In [ ]:
from pathlib import Path

checkpoint = Path('checkpoints/model_best.pt')
k12_ref = Path('data/reference/ecoli_k12_mg1655.fasta')
zymo_ref = Path('data/reference/ecoli_zymo_benchmark_strain.fasta')

assert checkpoint.exists(), 'checkpoints/model_best.pt not found -- ensure model_best.pt is inside checkpoints/'
assert k12_ref.exists(), 'K-12 reference not found at data/reference/ecoli_k12_mg1655.fasta'
assert zymo_ref.exists(), 'Zymo benchmark reference not found at data/reference/ecoli_zymo_benchmark_strain.fasta'

print(f'{checkpoint}: {checkpoint.stat().st_size:,} bytes -- OK')
print(f'{k12_ref}: {k12_ref.stat().st_size:,} bytes -- OK')
print(f'{zymo_ref}: {zymo_ref.stat().st_size:,} bytes -- OK')

In [ ]:
ckpt_meta = torch.load(checkpoint, map_location="cpu", weights_only=False)
missing = [k for k in ("model_state_dict", "model_config") if k not in ckpt_meta]
assert not missing, f"Checkpoint missing expected key(s) {missing} -- not written by save_checkpoint()?"
print("[OK] Checkpoint structurally matches the save_checkpoint()/load_from_checkpoint() contract.")
print(f"[INFO] model_config: {ckpt_meta['model_config']}")
print("[NOTE] This checkpoint format doesn't store epoch/val_loss (that's save_training_state()'s "
      "job, a separate file). The real check that this is the epoch-2 checkpoint is Part A.1's "
      "evaluate.py run below -- mean_identity should land near 0.7342.")
del ckpt_meta

## 3. Sanity Gate

In [ ]:
!find . -name '*.py' -exec python -m py_compile {} \;
!python -m pytest tests/ -q

## PART A.1 -- Real accuracy numbers: `evaluate.py`

Generating fresh synthetic evaluation reads using `--seed 999` and running free-running inference evaluation.

In [ ]:
import subprocess

subprocess.run([
    'python', '-m', 'data.simulator',
    '--reference', 'data/reference/ecoli_k12_mg1655.fasta',
    '--output', 'data/eval_pairs.jsonl',
    '--quantity', '2x',
    '--seed', '999',
], check=True)

In [ ]:
subprocess.run([
    'python', '-u', '-m', 'training.evaluate',
    '--checkpoint', 'checkpoints/model_best.pt',
    '--validation-data', 'data/eval_pairs.jsonl',
    '--output-dir', 'evaluation_output',
    '--max-examples', '100',
], check=True)

## PART A.2 -- Variant Preservation: `variant_preservation_test.py`

Testing reference-hallucination mitigation by evaluating planted mutations.

In [ ]:
subprocess.run([
    'python', '-u', '-m', 'training.variant_preservation_test',
    '--checkpoint', 'checkpoints/model_best.pt',
    '--reference', 'data/reference/ecoli_k12_mg1655.fasta',
    '--num-chunks', '50',
    '--mutations-per-chunk', '3',
], check=True)

---
## PART B -- Full baseline comparison: Ours vs. Racon/Medaka

### B.1 -- Verify CLI Tools for Part B

In [ ]:
for tool in ['flye', 'racon', 'medaka_consensus', 'assess_assembly', 'assess_homopolymers',
             'dnadiff', 'rasusa', 'samtools', 'minimap2']:
    check_cli_tool(tool)

### B.2 -- Get Zymo E. coli reads: real (from ENA) if possible, synthetic as fallback

In [ ]:
import shutil

total, used, free = shutil.disk_usage(".")
free_gb = free / (1024**3)
print(f"[INFO] Free disk space: {free_gb:.1f} GB")
if free_gb < 20:
    print("[WARNING] Under 20 GB free. The raw Zymo D6300 download can be several GB before "
          "subsetting -- this is what caused the Kaggle OSError last time. Consider freeing "
          "space or resizing your WSL virtual disk before proceeding.")

In [ ]:
import subprocess, os

ena_url_result = subprocess.run(
    ['curl', '-s', 'https://www.ebi.ac.uk/ena/portal/api/filereport?accession=ERR7287988&result=read_run&fields=fastq_ftp&format=tsv'],
    capture_output=True, text=True
)
print('ENA API response:')
print(ena_url_result.stdout)

ftp_url = None
lines = [l for l in ena_url_result.stdout.strip().split('\n') if l and not l.startswith('run_accession')]
if lines:
    candidate = lines[0].split('\t')[-1].split(';')[0].strip()
    if candidate.startswith('ftp.'):
        ftp_url = 'https://' + candidate

os.makedirs('benchmark_data', exist_ok=True)

if ftp_url:
    print(f'\nResolved real download URL: {ftp_url}')
    subprocess.run(['wget', '-q', '-O', 'benchmark_data/zymo_d6300_raw.fastq.gz', ftp_url], check=True)
    print('Downloaded real Zymo D6300 R10.4.1 reads.')
    USE_REAL_ZYMO = True
else:
    print('\nCould not resolve a real download URL automatically.')
    print('Falling back to SYNTHETIC Zymo-strain reads.')
    USE_REAL_ZYMO = False

In [ ]:
if not USE_REAL_ZYMO:
    subprocess.run([
        'python', '-m', 'data.simulator',
        '--reference', 'data/reference/ecoli_zymo_benchmark_strain.fasta',
        '--output', 'benchmark_data/zymo_synthetic_pairs.jsonl',
        '--quantity', '50x',
        '--seed', '2026',
    ], check=True)
    import json
    with open('benchmark_data/zymo_synthetic_pairs.jsonl') as f_in, open('benchmark_data/subset_50x.fastq', 'w') as f_out:
        for i, line in enumerate(f_in):
            rec = json.loads(line)
            seq = rec['noisy_sequence']
            f_out.write(f'@synthetic_read_{i}\n{seq}\n+\n{"I" * len(seq)}\n')
    print('Synthetic subset written to benchmark_data/subset_50x.fastq.')

### B.3 -- If using REAL Zymo reads: extract E. coli + downsample

In [ ]:
if USE_REAL_ZYMO:
    !chmod +x benchmarking/prepare_zymo_subset.sh
    subprocess.run([
        './benchmarking/prepare_zymo_subset.sh',
        'benchmark_data/zymo_d6300_raw.fastq.gz',
        'data/reference/ecoli_zymo_benchmark_strain.fasta',
        'benchmark_data',
        '50',
        '4.8m',
    ], check=True)
else:
    print('Skipped -- using synthetic subset.')

### B.4 -- Run the full comparison

In [ ]:
!chmod +x benchmarking/run_comparison.sh
subprocess.run([
    './benchmarking/run_comparison.sh',
    'benchmark_data/subset_50x.fastq',
    'data/reference/ecoli_zymo_benchmark_strain.fasta',
    'checkpoints/model_best.pt',
    'benchmark_data/results',
], check=True)